<img src="../../../img/logo-bdc.png" align="right" width="64"/>

# <span style="color:#336699"> Tile Map Service (BDC-Tiler)</span>
<hr style="border:2px solid #0077b9;">

BDC-Tiler is a Tile Map Service, created to serve the data products generated by the Brazil Data Cube project. It renders tiles on the fly from any Cloud Optimized GeoTIFF (COG), which is passed through the `url` parameter. This document presents an example of BDC-Tiler consumption using [folium](https://python-visualization.github.io/folium/), with the images being retrieved from the [BDC STAC](https://data.inpe.br/bdc/stac/v1/).

To begin, first install the packages that will be used.

In [ ]:
!pip install folium pystac-client > /dev/null

With the packages installed, load `folium`, used to display the map, and `pystac_client`, used to search the images that will be displayed.

In [ ]:
import folium
import pystac_client

Now the declaration of the variables used in this example is made. The variable `url_service` is the BDC-Tiler endpoint, using the `WebMercatorQuad` tiling scheme. The `{z}`, `{x}` and `{y}` placeholders are filled by the map client with the zoom level and the tile coordinates.

In [ ]:
stac_url = "https://data.inpe.br/bdc/stac/v1/"
url_service = "https://data.inpe.br/bdc/tms/tiles/WebMercatorQuad/{z}/{x}/{y}"

Once this is done, search for an image over the area of interest and select the asset that will be displayed. The asset must be an RGB COG, ready to be visualized, such as the `TCI` (True Color Image) of Sentinel-2 or the `VISUAL` of the CBERS-4 mosaic.

> To learn more about searching images, see [stac-introduction](https://github.com/brazil-data-cube/code-gallery/blob/master/jupyter/Python/stac/stac-introduction.ipynb).

In [ ]:
catalog = pystac_client.Client.open(stac_url)

center = (-12.50, -45.75)  # (lat, lon)
point = {"type": "Point", "coordinates": [center[1], center[0]]}  # [lon, lat]

# For Sentinel 2 use:
collection = "S2_L2A-1"
datetime = "2024-08-01/2024-08-31"
asset = "TCI"

# For CBERS use the following values
# collection = "mosaic-cbers4-brazil-3m-1"
# datetime = "2020-04-01/2020-06-30"
# asset = "VISUAL"

item_search = catalog.search(
    collections=[collection],
    intersects=point,
    datetime=datetime,
    sortby="properties.eo:cloud_cover",
    max_items=1
)

item = next(item_search.items())
cog = item.assets[asset]

print(item.id)
print(cog.href)

There you go, now add the content to the screen through a `folium.Map` and a `folium.TileLayer`, passing the COG address in the `url` parameter of BDC-Tiler.

In [ ]:
m = folium.Map(location=center, zoom_start=10)

scene_layer = folium.TileLayer(
    tiles=f"{url_service}?url={cog.href}",
    attr="Brazil Data Cube - INPE",
    name=item.id,
    overlay=True,
    control=True
)

# add layer in the map
scene_layer.add_to(m)
folium.LayerControl().add_to(m)

m

## Contrast enhancement

BDC-Tiler also accepts parameters to enhance the image on the fly. The `color_formula` parameter applies a sequence of operations, such as `Gamma`, `Sigmoidal` and `Saturation`, to the RGB bands. Like any query parameter, it must be URL encoded.

> Change the values of `color_formula` to adjust the contrast. Examples: `"Gamma RGB 1.5"`, `"Sigmoidal RGB 6 0.4 Saturation 1.3"` or `"Gamma RGB 2 Saturation 1.8 Sigmoidal RGB 5 0.35"`.

In [ ]:
from urllib.parse import quote_plus

# quote as url encoded
color_formula = quote_plus("Sigmoidal RGB 6 0.4 Saturation 1.3")

m = folium.Map(location=center, zoom_start=12)

original_layer = folium.TileLayer(
    tiles=f"{url_service}?url={cog.href}",
    attr="Brazil Data Cube - INPE",
    name=f"{item.id} (original)",
    overlay=True,
    control=True
)

enhanced_layer = folium.TileLayer(
    tiles=f"{url_service}?url={cog.href}&color_formula={color_formula}",
    attr="Brazil Data Cube - INPE",
    name=f"{item.id} (color_formula)",
    overlay=True,
    control=True
)

original_layer.add_to(m)
enhanced_layer.add_to(m)
folium.LayerControl().add_to(m)

m

Another option is the `rescale` parameter, which performs a linear stretch, mapping the interval `min,max` of the pixel values to `0,255`. Lower values of `max` result in a brighter image.

In [ ]:
rescale = "0,128"  # min,max

m = folium.Map(location=center, zoom_start=12)

rescaled_layer = folium.TileLayer(
    tiles=f"{url_service}?url={cog.href}&rescale={rescale}",
    attr="Brazil Data Cube - INPE",
    name=f"{item.id} (rescale {rescale})",
    overlay=True,
    control=True
)

rescaled_layer.add_to(m)
folium.LayerControl().add_to(m)

m